In [2]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class LactamFormation(MorphingOperator):
    def __init__(self):
        super(LactamFormation, self).__init__()
        self._name = "Lactam Formation (Phase I - Ring Oxidation Safe)"
        self._matches = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;R;!$(N-C=O);!a]-[CX4;R;H2;!$(C-O)]")

    def setOriginal(self, mol):
        super(LactamFormation, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        n_idx, c_idx = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            o_atom = Chem.Atom(8)
            o_atom.SetFormalCharge(0)
            o_idx = rw_mol.AddAtom(o_atom)
            
            rw_mol.AddBond(c_idx, o_idx, Chem.BondType.DOUBLE)
            
            for idx in [n_idx, c_idx, o_idx]:
                atom = rw_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)
            
            new_mol = rw_mol.GetMol()
            
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

lactam_op = LactamFormation()

print("STARTING LACTAM FORMATION TREE SIMULATION")
print("=========================================================")
# Αρχικό Μόριο: Πυρρολιδίνη
root_mol = MolpherMol("C1CCNC1")
print(f"GENERATION 0 (Pyrrolidine Root):\n  SMILES: {root_mol.getSMILES()}\n")

# --- ΓΕΝΙΑ 1: Οξείδωση σε Λακτάμη ---
lactam_op.setOriginal(root_mol)
gen1_mol = lactam_op.morph()
print(f"GENERATION 1 (Oxidation to Lactam):")
print(f"  SOURCE: {root_mol.getSMILES()}")
print(f"  TARGET: {gen1_mol.getSMILES() if gen1_mol else 'Failed'}")
print("\n")

# --- ΓΕΝΙΑ 2: Προσπάθεια για δεύτερη οξείδωση (Πρέπει να αποκλειστεί!) ---
lactam_op.setOriginal(gen1_mol)
gen2_mol = lactam_op.morph()
print(f"GENERATION 2 (Anti-Overoxidation Protection Check):")
print(f"  SOURCE: {gen1_mol.getSMILES()}")
print(f"  TARGET: {gen2_mol.getSMILES() if gen2_mol else 'Failed'}")

if gen1_mol.getSMILES() == gen2_mol.getSMILES():
    print("  PROTECTION CHECK: Passed (Successfully blocked overoxidation into chemical monsters!)")
print("=========================================================")
# Το επίσημο SMILES της Νικοτίνης
nicotine_smiles = "CN1CCCC1c2cccnc2"
nicotine_mol = MolpherMol(nicotine_smiles)

# Χρήση του θωρακισμένου operator που φτιάξαμε
lactam_op = LactamFormation()
lactam_op.setOriginal(nicotine_mol)
cotinine_mol = lactam_op.morph()

print("NICOTINE TO COTININE METABOLISM BENCH")
print(f"SOURCE (Nicotine): {nicotine_mol.getSMILES()}")
print(f"TARGET (Cotinine): {cotinine_mol.getSMILES() if cotinine_mol else 'Failed'}")
cotinine_pattern = Chem.MolFromSmarts("[N;R]([CH3])[C](=O)")
if cotinine_mol and cotinine_mol.asRDMol().HasSubstructMatch(cotinine_pattern):
    print("METABOLIC ACCURACY: Passed!")
else:
    print("METABOLIC ACCURACY: Failed.")

STARTING LACTAM FORMATION TREE SIMULATION
GENERATION 0 (Pyrrolidine Root):
  SMILES: C1CCNC1

GENERATION 1 (Oxidation to Lactam):
  SOURCE: C1CCNC1
  TARGET: O=C1CCCN1


GENERATION 2 (Anti-Overoxidation Protection Check):
  SOURCE: O=C1CCCN1
  TARGET: O=C1CCCN1
  PROTECTION CHECK: Passed (Successfully blocked overoxidation into chemical monsters!)
NICOTINE TO COTININE METABOLISM BENCH
SOURCE (Nicotine): CN1CCCC1C1=CC=CN=C1
TARGET (Cotinine): CN1C(=O)CCC1C1=CC=CN=C1
METABOLIC ACCURACY: Passed!
